In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("dataset_100pc(in).csv")
df["parallax_"] = df["parallax"]/1000
df["relative_error"] = df["parallax_error"]/df["parallax"]
df = df[df["relative_error"]<0.1]

coma_ra = 185.625
coma_dec = 25.85
coma_dist = 86 
coma_deg_window = 15
coma_parallax_window = 20

coma_conditions = (1/(coma_dist+coma_parallax_window) <= df["parallax_"]) & (df["parallax_"] <= 1/(coma_dist-coma_parallax_window)) & ((coma_dec-coma_deg_window) <= df["dec"]) & (df["dec"] <= coma_dec+coma_deg_window) & (coma_ra-coma_deg_window <= df["ra"]) & (df["ra"] <= coma_ra+coma_deg_window)

hyades_ra = 66.75
hyades_dec = 15.867
hyades_dist = 46 
hyades_deg_window = 15
hyades_parallax_window = 20


hyades_conditions = (1/(hyades_dist+hyades_parallax_window) <= df["parallax_"]) & (df["parallax_"] <= 1/(hyades_dist-hyades_parallax_window)) & ((hyades_dec-hyades_deg_window) <= df["dec"]) & (df["dec"] <= hyades_dec+hyades_deg_window) & (hyades_ra-hyades_deg_window <= df["ra"]) & (df["ra"] <= hyades_ra+hyades_deg_window)

cluster_mask = (hyades_conditions | coma_conditions)
df = df[~cluster_mask]

In [2]:
# Crear bins de la grilla 8x8
ra_bins = np.linspace(df["ra"].min(), df["ra"].max(), 9)
dec_bins = np.linspace(df["dec"].min(), df["dec"].max(), 9)

df["grid_i"] = np.digitize(df["ra"], ra_bins) - 1
df["grid_j"] = np.digitize(df["dec"], dec_bins) - 1

# Clip para evitar índices fuera de rango en los bordes
df["grid_i"] = df["grid_i"].clip(0, 7)
df["grid_j"] = df["grid_j"].clip(0, 7)

df["cell_id"] = df["grid_i"] * 8 + df["grid_j"] + 1  # 1 a 64

df = df.drop(columns=["grid_i", "grid_j"])

In [3]:
df.to_csv("dataset_porcionado.csv")